# 01 — Розвідувальний аналіз даних (EDA)

**Студент:** Олександр Щепанчук &nbsp;&nbsp; **Група:** ПМПм-12

**Датасет:** [SNAP Reddit Hyperlink Network](https://snap.stanford.edu/data/soc-RedditHyperlinks.html)

Each example below is an *observed* directed subreddit-to-subreddit hyperlink. We classify the sentiment label attached to that hyperlink (`POST_LABEL`), remapped as `-1 -> 0` (negative) and `+1 -> 1` (neutral/positive). **No negative sampling. Label 0 is the negative class, not a non-edge.**

Every figure cell saves a PNG under `reports/figures/` and prints the path so the submission notebook can embed it without recomputing.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from reddit_gnn.paths import PATHS

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

EDGES_PATH = PATHS.data_processed / 'edges.parquet'
META_PATH = PATHS.data_processed / 'metadata.json'
N2I_PATH = PATHS.data_processed / 'node_to_id.json'
FIG_DIR = PATHS.reports_figures
TBL_DIR = PATHS.reports_tables

def show_image_or_msg(path: Path, label: str = '') -> None:
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path)))
        print(f'shown: {path}')
    else:
        print(f'Missing — run `make all` first: {path}')

def load_table_or_msg(path: Path, **read_kwargs) -> pd.DataFrame | None:
    path = Path(path)
    if not path.exists():
        print(f'Missing — run `make all` first: {path}')
        return None
    return pd.read_csv(path, **read_kwargs)

## Завантаження оброблених даних

We load the parquet produced by `scripts/prepare_data.py`. If it's missing, every subsequent cell prints a clear *Run `make all` first* message and exits.

In [2]:
if not EDGES_PATH.exists():
    print(f'Missing — run `make all` first: {EDGES_PATH}')
    df = None
    metadata = None
else:
    df = pd.read_parquet(EDGES_PATH)
    metadata = json.loads(META_PATH.read_text())
    print(f'Loaded {len(df):,} edges, {metadata["num_nodes"]:,} nodes')
    display(df.head())

Missing — run `make all` first: /mnt/d/oleksandr/GNN/data/processed/edges.parquet


## Огляд статистик / Basic graph statistics

In [3]:
from reddit_gnn.analysis.graph_stats import compute_basic_stats
if df is None:
    print('Run `make all` first.')
else:
    basic = compute_basic_stats(df)
    display(pd.DataFrame.from_records([basic]).T.rename(columns={0: 'value'}))

Run `make all` first.


## Розподіл міток / Edge label distribution

Class balance over the trained labels (0 = negative, 1 = neutral/positive).

In [4]:
from reddit_gnn.analysis.signed_stats import compute_label_stats
from reddit_gnn.visualization.distributions import plot_label_distribution
if df is None:
    print('Run `make all` first.')
    label_stats = None
else:
    label_stats = compute_label_stats(df)
    print(label_stats)
    out = FIG_DIR / '01_label_distribution.png'
    plot_label_distribution(df, save_path=out)
    plt.show(); print(f'saved: {out}')

Run `make all` first.


## Розподіл степенів / In- and out-degree distributions (log-log)

Heavy-tailed in-degree is the norm for Reddit-style graphs — a few hub subreddits attract orders-of-magnitude more activity than the median.

In [5]:
from reddit_gnn.analysis.graph_stats import compute_degree_stats
from reddit_gnn.visualization.distributions import plot_degree_distribution
if df is None:
    print('Run `make all` first.')
    deg = None
else:
    deg = compute_degree_stats(df)
    print('summary:', deg['summary'])
    for kind in ('in', 'out', 'total'):
        out = FIG_DIR / f'01_degree_{kind}_log.png'
        plot_degree_distribution(deg[f'{kind}_degree'], degree_type=kind, log_scale=True, save_path=out)
        plt.show(); print(f'saved: {out}')

Run `make all` first.


## Топ сорсів / таргетів за степенем

In [6]:
from reddit_gnn.visualization.distributions import plot_top_subreddits_by_degree
if df is None:
    print('Run `make all` first.')
else:
    for mode in ('total', 'in', 'out'):
        out = FIG_DIR / f'01_top_{mode}_degree.png'
        plot_top_subreddits_by_degree(df, top_k=20, mode=mode, save_path=out)
        plt.show(); print(f'saved: {out}')

Run `make all` first.


## Реципрокність і компоненти зв'язності

Reciprocity = fraction of `(u, v)` directed edges whose reverse `(v, u)` also exists. Components come from `scipy.sparse.csgraph` over a sparse binary adjacency.

In [7]:
from reddit_gnn.analysis.graph_stats import compute_component_stats, compute_reciprocity_stats
if df is None:
    print('Run `make all` first.')
    recip = None; comp = None
else:
    recip = compute_reciprocity_stats(df)
    comp = compute_component_stats(df)
    print('reciprocity:', recip)
    print('components:', comp)

Run `make all` first.


## Динаміка обсягу / Edges over time

Volume per month, stacked by sign — motivates the chronological train/val/test split used downstream.

In [8]:
from reddit_gnn.visualization.temporal import plot_edges_over_time
if df is None:
    print('Run `make all` first.')
else:
    out = FIG_DIR / '01_edges_over_time.png'
    plot_edges_over_time(df, freq='ME', save_path=out)
    plt.show(); print(f'saved: {out}')

Run `make all` first.


## Дрейф негативної частки / Negative-label share over time

In [9]:
from reddit_gnn.visualization.temporal import plot_negative_ratio_over_time
if df is None:
    print('Run `make all` first.')
else:
    out = FIG_DIR / '01_negative_ratio_over_time.png'
    plot_negative_ratio_over_time(df, freq='ME', save_path=out)
    plt.show(); print(f'saved: {out}')

Run `make all` first.


## Випадковий підграф / Sampled signed subgraph

300-edge uniform sample drawn with red = negative, green = neutral/positive, arrows for direction.

In [10]:
from reddit_gnn.visualization.subgraphs import plot_sampled_signed_subgraph
if df is None:
    print('Run `make all` first.')
else:
    out = FIG_DIR / '01_sampled_signed_subgraph.png'
    plot_sampled_signed_subgraph(df, max_edges=300, seed=42, save_path=out)
    plt.show(); print(f'saved: {out}')

Run `make all` first.


## Ego networks для миролюбного і ворожого сабреддітів

*Peaceful* = lowest negative outgoing-link ratio; *hostile* = highest. Both filtered to subreddits with at least 50 outgoing edges so the ratio is stable.

In [11]:
from reddit_gnn.analysis.signed_stats import negative_ratio_by_source
from reddit_gnn.visualization.subgraphs import plot_ego_signed_subgraph

if df is None:
    print('Run `make all` first.')
else:
    grouped = df.groupby('source_subreddit_norm')['label_binary'].agg(['count', 'mean']).reset_index()
    grouped['negative_ratio'] = 1.0 - grouped['mean']
    grouped = grouped[grouped['count'] >= 50]
    hostile = grouped.sort_values('negative_ratio', ascending=False).iloc[0]['source_subreddit_norm']
    peaceful = grouped.sort_values('negative_ratio', ascending=True).iloc[0]['source_subreddit_norm']
    print(f'peaceful: {peaceful}  |  hostile: {hostile}')
    for label, sub in (('peaceful', peaceful), ('hostile', hostile)):
        out = FIG_DIR / f'01_ego_{label}.png'
        plot_ego_signed_subgraph(df, subreddit=sub, radius=1, max_edges=300, seed=42, save_path=out)
        plt.show(); print(f'saved {label} ego ({sub}): {out}')

Run `make all` first.


## Підсумки EDA / Wrap-up

Single-row summary CSV mirrors what gets reported in the submission notebook. Numbers are written from the live `df`; if no data is loaded the file is not produced.

In [12]:
from reddit_gnn.analysis.signed_stats import signed_triad_counts
if df is None:
    print('Run `make all` first — stats_summary.csv will not be written.')
else:
    triads = signed_triad_counts(df, sample_cap=50_000)
    summary = {
        **basic,
        **label_stats,
        'reciprocity': recip['reciprocity'],
        'wcc_count': comp['wcc_count'],
        'largest_wcc_size': comp['largest_wcc_size'],
        'scc_count': comp['scc_count'],
        'largest_scc_size': comp['largest_scc_size'],
        'triads_balanced': triads['balanced'],
        'triads_unbalanced': triads['unbalanced'],
        'time_min': metadata['timestamp_range']['min'],
        'time_max': metadata['timestamp_range']['max'],
    }
    out = TBL_DIR / 'stats_summary.csv'
    out.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([summary]).to_csv(out, index=False)
    print(f'wrote {out}')
    display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))

Run `make all` first — stats_summary.csv will not be written.


---

EDA complete. The submission notebook `02_main_experiment.ipynb` embeds the saved figures and table from this run.